# Day 01 — LLM Lifecycle: Tokenization, Parameters and Forward Pass

**Curriculum:** O01 + Stanford CME295 S1  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Environment:** Local CPU  
**Status:** Practical in progress

## Learning objectives
1. Inspect tokenization and token IDs.
2. Load a causal language model and inspect its parameters.
3. Interpret input and logits tensor shapes.
4. Inspect EOS and choose a next token using greedy decoding.
5. Later: inspect candidate logits, generate a short continuation, review the model card, resolve doubts and complete the mock interview.

**Mentoring sequence:** Theory → Theory doubts → Practical → Practical doubts → Mock interview. Write and verify one step at a time.

**Stanford S1:** [Lecture 1](https://www.youtube.com/watch?v=Ub3GoFaUcds) — tokenization from 22:57; word representations from 30:28. Record what you actually watched.

## 0. Setup

Install in your existing `uv` environment (terminal):

```powershell
uv add --dev ipykernel jupyter
```

Use VS Code's **Select Kernel** to choose the project's `.venv` Python interpreter. Required runtime packages: `torch`, `transformers` (already installed in the Day 1 environment). Internet access may be needed for the first model download.

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

## 1. Theory — Tokens and embeddings

A tokenizer converts text to token IDs. An embedding matrix contains a learned vector for each vocabulary entry. For this model, the observed first parameter has shape `(49152, 576)`: 49,152 vocabulary entries × 576 values per embedding = **28,311,552** embedding parameters. This is only one parameter tensor, not the total model parameter count.

## 2. Practical — Tokenization (completed)

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(type(tokenizer).__name__)

text = "I am polishing my AI Engineeering"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text, add_special_tokens=False)

print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Token count:", len(token_ids))
print("Decoded:", tokenizer.decode(token_ids))

GPT2Tokenizer
Tokens: ['I', 'Ġam', 'Ġpolishing', 'Ġmy', 'ĠAI', 'ĠEngine', 'eering']
Token IDs: [57, 744, 39493, 957, 5646, 6059, 12212]
Token count: 7
Decoded: I am polishing my AI Engineeering


## 3. Practical — Load model on CPU (completed)

In [15]:
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.to("cpu")
model.eval()

print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model class: LlamaForCausalLM
Device: cpu


## 4. Practical — Parameter inspection (code completed; verify output)

`numel()` counts scalar elements; `named_parameters()` yields names and tensors. `detach()` separates the inspection tensor from autograd. `model.eval()` selects evaluation behavior but does not itself disable gradient tracking.

In [16]:
parameter_count = sum(p.numel() for p in model.parameters())
name, parameter = next(model.named_parameters())

print("Total parameters:", f"{parameter_count:,}")
print("First parameter name:", name)
print("Shape:", tuple(parameter.shape))
print("Data type:", parameter.dtype)
print("First five values:", parameter.detach().reshape(-1)[:5])
print("Embedding parameters:", parameter.numel())

Total parameters: 134,515,008
First parameter name: model.embed_tokens.weight
Shape: (49152, 576)
Data type: torch.bfloat16
First five values: tensor([-0.1060,  0.0342,  0.0476, -0.0239, -0.0674], dtype=torch.bfloat16)
Embedding parameters: 28311552


## 5. Theory — Forward pass and logits

For batch size **B**, sequence length **T**, and vocabulary size **V**, logits have shape `(B, T, V)`. A causal model predicts a distribution for the next token at each position. The final position's scores are used to select the next generated token. `torch.inference_mode()` avoids tracking gradients during this inference.

## 6. Practical — Forward pass and logits (completed code)

In [17]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False,
)

with torch.inference_mode():
    outputs = model(**inputs)

print("Input IDs Shape:", tuple(inputs["input_ids"].shape))
print("Logits shape:", tuple(outputs.logits.shape))

next_token_logits = outputs.logits[:, -1, :]
print("Next-token logits shape:", tuple(next_token_logits.shape))

Input IDs Shape: (1, 7)
Logits shape: (1, 7, 49152)
Next-token logits shape: (1, 49152)


## 7. Practical — EOS inspection (completed code)

In [18]:
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("Generation EOS ID:", model.generation_config.eos_token_id)

EOS token: <|im_end|>
EOS token ID: 2
Generation EOS ID: 2


## 8. Practical — Greedy next-token selection (completed code)

`argmax` finds the highest-scoring vocabulary entry. Softmax preserves score ordering, so greedy selection does not require computing probabilities first.

In [19]:
next_token_id = next_token_logits.argmax(dim=-1)

print("Predicted token ID:", next_token_id.item())
print("Predicted token:", tokenizer.decode(next_token_id))

Predicted token ID: 30
Predicted token: .


## 9. Next exercise — Top five candidate tokens (not yet completed)

**Your task:** Inspect the five highest logits and decode their token IDs. Record the results, then answer: *If the top logit is 8.5, does that imply 85% probability? Why or why not?*

Write the code yourself below after our next discussion.

In [20]:
top_logits, top_ids = torch.topk(
    next_token_logits,
    k=5,
    dim = -1,
)

for tokken_id, logit in zip(top_ids[0], top_logits[0]):
    print(
        repr(tokenizer.decode([tokken_id.item()])),
        round(logit.item(), 3)
    )


'.' 17.75
' and' 17.625
',' 17.625
' to' 17.5
' skills' 17.375


In [21]:
# Convert all vocabulary logits into probabilities
probabilities = torch.softmax(next_token_logits, dim=-1)

# Get the top 5 probabilities and corresponding token IDs
top_probs, top_ids = torch.topk(probabilities, k=5, dim=-1)

for token_id, prob in zip(top_ids[0], top_probs[0]):
    token = tokenizer.decode([token_id.item()])
    print(f"{repr(token)}: {prob.item():.4%}")

print("Total probabilitiy: ", probabilities.sum().item())

'.': 3.9795%
' and': 3.5156%
',': 3.5156%
' to': 3.1128%
' skills': 2.7466%
Total probabilitiy:  1.0


## 11. Autoregressive CPU generation

**Previously clarified:** vocabulary × embedding dimensions; batch/sequence/vocabulary axes; EOS supervision; autoregressive stopping; greedy selection from logits.

**Your observations (fill after execution):**
- Expected:
- Observed:
- Why they differ (if applicable):
- Stanford S1 watched timestamps and takeaway:
- Remaining questions:


In [22]:
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

input_length = inputs["input_ids"].shape[1]
new_token_ids = generated_ids[0, input_length:]

print("Input tokens:", input_length)
print("Generated tokens:", len(new_token_ids))
print("Generated text:", tokenizer.decode(
    new_token_ids,
    skip_special_tokens=True,
))

print("Full text:", tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
))

Input tokens: 7
Generated tokens: 20
Generated text: . I am a master of the art of the machine. I am a master of the machine.
Full text: I am polishing my AI Engineeering. I am a master of the art of the machine. I am a master of the machine.


In [23]:
import transformers

print("Model ID:", model_id)
print("Model revision:", getattr(model.config, "_commit_hash", None))
print("Prompt:", repr(text))
print("Input token count:", inputs["input_ids"].shape[1])
print("Generated token count:", len(new_token_ids))
print("Device:", next(model.parameters()).device)
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

Model ID: HuggingFaceTB/SmolLM2-135M-Instruct
Model revision: None
Prompt: 'I am polishing my AI Engineeering'
Input token count: 7
Generated token count: 20
Device: cpu
Torch version: 2.14.1+cpu
Transformers version: 5.19.0


## 12. Mock interview — pending

We will do this interactively **after** the practical and doubts are complete, one question at a time.

Opening question: **Which operations change activations, and which can update weights? What evidence from your run supports your answer?**

- My answer:
- Mentor feedback (0–3):
- Retry:

## 13. Session evidence / completion gate

- [x] Theory discussed
- [x] Initial theory doubts discussed (may reopen)
- [ ] Practical verified in notebook
- [ ] Practical doubts cleared
- [ ] Mock interview completed

**Repository:** `topics/01-llm-lifecycle/practical/day01_llm_lifecycle.ipynb`  
**Next exact step:** execute the existing cells in order; then implement the top-five-logits exercise yourself.